In [18]:
import os
import numpy as np
import torch.nn as nn
from sklearn.metrics import classification_report,confusion_matrix
import laspy
import open3d as o3d
from torch.utils.data import DataLoader,Dataset,Subset
import torch.nn.functional as F
import torch
import mlflow
from sklearn.model_selection import train_test_split,KFold

In [19]:
class WoodPowderDataset(Dataset):
    def __init__(self,filepaths,num_points=1024):
        super().__init__()
        self.filepaths=filepaths
        self.num_points=num_points

    def __len__(self, other):
        return len(self.filepaths)
    
    def load_point_cloud(self,filepath):
        ext=filepath.split(".")[-1]
        if ext in [".laz",".las"]:
            las=laspy.read(filepath)
            xyz=np.column_stack((las.x,las.y,las.z))
        elif ext==".pcd":
            pcd=o3d.io.read_point_cloud(filepath)
            xyz=np.asarray(pcd.points)
        else:
            print("Unsupported file format")

    def __getitem__(self, index):
        filepath=self.filepaths[index]
        las=laspy.read(filepath)
        points=np.column_stack((las.x,las.y,las.z))
        labels=np.array(las.classification)
        if len(points) >= self.num_points:
            choice = np.random.choice(len(points), self.num_points, replace=False)
        else:
            
            choice = np.random.choice(len(points), self.num_points, replace=True)
            
        sampled_points = points[choice, :]
        sampled_points = sampled_points - np.mean(sampled_points, axis=0)
        sampled_labels = labels[choice] 
        max_dist = np.max(np.sqrt(np.sum(sampled_points**2, axis=1)))
        if max_dist > 0:
            sampled_points = sampled_points / max_dist
       
        point_tensor = torch.tensor(sampled_points, dtype=torch.float32)
        # .transpose(0, 1)
        label_tensor = torch.tensor(sampled_labels, dtype=torch.long)
        
        return point_tensor, label_tensor

In [20]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [21]:
class PointNetSegmentation(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Local features
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)

        # Segmentation specific MLPs (Combining Global & Local)
        self.conv4 = nn.Conv1d(1088, 512, 1) # 1024 (global) + 64 (local)
        self.conv5 = nn.Conv1d(512, 256, 1)
        self.conv6 = nn.Conv1d(256, 128, 1)
        self.conv7 = nn.Conv1d(128, num_classes, 1)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)
        self.bn6 = nn.BatchNorm1d(128)

    def forward(self, x):
        n_pts = x.size(2)
        
        # Extract Local features
        local_feat = F.relu(self.bn1(self.conv1(x)))      # [Batch, 64, N]
        x = F.relu(self.bn2(self.conv2(local_feat)))      
        x = F.relu(self.bn3(self.conv3(x)))               # [Batch, 1024, N]

        # Extract Global features
        global_feat = torch.max(x, 2, keepdim=True)[0]    # [Batch, 1024, 1]

        # Concatenate Local and Global features
        global_feat_expanded = global_feat.repeat(1, 1, n_pts) 
        concat_feat = torch.cat([local_feat, global_feat_expanded], 1) # [Batch, 1088, N]

        # Final Segmentation Layers
        x = F.relu(self.bn4(self.conv4(concat_feat)))     
        x = F.relu(self.bn5(self.conv5(x)))               
        x = F.relu(self.bn6(self.conv6(x)))               
        out = self.conv7(x)                               # Output: [Batch, 2, N]

        return out

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_cluster import knn_graph

class PointTransformerLayer(nn.Module):
    def __init__(self, in_channels, out_channels, k=16):
        super().__init__()
        self.k = k
        self.linear_q = nn.Linear(in_channels, out_channels)
        self.linear_k = nn.Linear(in_channels, out_channels)
        self.linear_v = nn.Linear(in_channels, out_channels)
        
        # Position encoding MLP
        self.pos_mlp = nn.Sequential(
            nn.Linear(3, out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels)
        )
        
        self.attn_mlp = nn.Sequential(
            nn.Linear(out_channels, out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels)
        )

    def forward(self, x, pos,batch=None):
        # x: (N, C), pos: (N, 3)
    
        edge_index = knn_graph(pos, self.k, batch=batch, loop=False)
        row, col = edge_index[0], edge_index[1]
        
        # Query, Key, Value
        q = self.linear_q(x)[row]  # (E, C)
        k = self.linear_k(x)[col]  # (E, C)
        v = self.linear_v(x)[col]  # (E, C)
        
        # Position Encoding
        pos_diff = pos[row] - pos[col]  # (E, 3)
        pos_enc = self.pos_mlp(pos_diff)  # (E, C)
        
        # Attention scores
        attn = self.attn_mlp(q - k + pos_enc)  # (E, C)
        attn = F.softmax(attn, dim=-1)
        
        # Aggregate
        agg = (attn * (v + pos_enc)).sum(dim=1, keepdim=False) # (N, C) - simplified aggregation
        
        # Note: For exact PyG scatter, use scatter_add. Here using simple loop/index for clarity.
        # In production, use torch_scatter.scatter_add
        out = torch.zeros_like(x)
        out.index_add_(0, row, attn * (v + pos_enc))
        
        return out

class PointTransformerSeg(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        # Encoder
        self.fc1 = nn.Linear(in_channels, 64)
        self.pt1 = PointTransformerLayer(64, 64, k=16)
        
        self.fc2 = nn.Linear(64, 128)
        self.pt2 = PointTransformerLayer(128, 128, k=16)
        
        # Decoder / Head
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)

    # def forward(self, pos, x=None):
    #     # pos shape: (Batch, N, 3)
    #     B, N, _ = pos.shape
        
    
    #     pos_flat = pos.reshape(-1, 3)
    #     batch = torch.arange(B, device=pos.device).repeat_interleave(N)
        
    #     if x is None:
    #         x = pos_flat
    #     else:
    #         x = x.reshape(-1, x.shape[-1])
            
    #     # Layer 1
    #     x = F.relu(self.fc1(x))
    #     x = self.pt1(x, pos_flat, batch) 
        
    #     # Layer 2
    #     x = F.relu(self.fc2(x))
    #     x = self.pt2(x, pos_flat, batch)
        
    #     # Head
    #     x = F.relu(self.fc3(x))
    #     x = self.fc4(x)
        
    
    #     return x.reshape(B, N, -1) 


    def forward(self, x, pos, batch=None):
        edge_index = knn_graph(pos, self.k, batch=batch, loop=False)
        row, col = edge_index[0], edge_index[1]

        q = self.linear_q(x)[row]
        k = self.linear_k(x)[col]
        v = self.linear_v(x)[col]

        pos_diff = pos[row] - pos[col]
        pos_enc = self.pos_mlp(pos_diff)

        attn = self.attn_mlp(q - k + pos_enc)  # (E, C)

        # Softmax must be over each query's neighbors, not over channels.
        # scatter_softmax normalizes per (row, channel) group across the edges
        # that share the same row (i.e. across the k neighbors).
        from torch_scatter import scatter_softmax, scatter_add
        attn = scatter_softmax(attn, row, dim=0)   # normalize across neighbors per point

        out = scatter_add(attn * (v + pos_enc), row, dim=0, dim_size=x.size(0))
        return out

In [23]:
# def train_evaluate_fold(fold, model, train_loader, val_loader, epochs=30, lr=0.001, device="cuda"):
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)
#     # criterion = nn.CrossEntropyLoss()
#     class_weights = torch.tensor([1.0, 10.0]).to(device) 
#     criterion = nn.CrossEntropyLoss(weight=class_weights)
#     with mlflow.start_run(nested=True, run_name=f"Fold-{fold+1}"):
#         for epoch in range(epochs):
#             # --- Training Phase ---
#             model.train()
#             train_loss, train_correct, train_total = 0.0, 0, 0
            
#             for points, labels in train_loader:
#                 points, labels = points.to(device), labels.to(device)
#                 optimizer.zero_grad()
                
#                 outputs = model(points) 
#                 # outputs = outputs.transpose(2, 1).contiguous().view(-1, 2)
#                 loss = criterion(outputs.view(-1, 2), labels.view(-1)) 
#                 loss.backward()
#                 optimizer.step()
                
#                 train_loss += loss.item() * points.size(0)
#                 _, predicted = torch.max(outputs.data, 2)
#                 train_total += labels.numel()
#                 train_correct += (predicted == labels).sum().item()
#                 # train_correct += (predicted == labels.view(-1)).sum().item()

#             epoch_train_loss = train_loss / len(train_loader.dataset)
#             epoch_train_acc = train_correct / train_total

#             # --- Validation Phase ---
#             model.eval()
#             val_loss, val_correct, val_total = 0.0, 0, 0
            
#             with torch.no_grad():
#                 for points, labels in val_loader:
#                     points, labels = points.to(device), labels.to(device)
#                     outputs = model(points)
#                     # outputs = model(points)
#                     # outputs = outputs.transpose(2, 1).contiguous().view(-1, 2)
#                     # loss = criterion(outputs, labels.view(-1))
#                     loss = criterion(outputs.view(-1, 2), labels.view(-1))
                    
#                     val_loss += loss.item() * points.size(0)
#                     _, predicted = torch.max(outputs.data, 1)
#                     val_total += labels.numel()
#                     val_correct += (predicted == labels).sum().item()
#                     # val_correct += (predicted == labels.view(-1)).sum().item()
                    
#             epoch_val_loss = val_loss / len(val_loader.dataset)
#             epoch_val_acc = val_correct / val_total
            
#             # MLflow Logging
#             mlflow.log_metrics({
#                 "train_loss": epoch_train_loss,
#                 "train_acc": epoch_train_acc,
#                 "val_loss": epoch_val_loss,
#                 "val_acc": epoch_val_acc
#             }, step=epoch)
            
#             print(f"Fold {fold+1} | Epoch {epoch+1}/{epochs} | "
#                   f"Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.4f} | "
#                   f"Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.4f}")
            
#     return model 

def train_evaluate_fold(fold, model, train_loader, val_loader, epochs=30, lr=0.001, device="cuda"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    class_weights = torch.tensor([1.0, 10.0]).to(device) 
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    with mlflow.start_run(nested=True, run_name=f"Fold-{fold+1}"):
        for epoch in range(epochs):
            # --- Training Phase ---
            model.train()
            train_loss, train_correct, train_total = 0.0, 0, 0
            
            for points, labels in train_loader:
                points, labels = points.to(device), labels.to(device)
                optimizer.zero_grad()
                
                outputs = model(points) # Shape: (Batch, N, 2)
                
                # Flatten for Loss
                loss = criterion(outputs.view(-1, 2), labels.view(-1))
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item() * points.size(0)
                
                # Predictions (dim=2 because shape is Batch x N x 2)
                _, predicted = torch.max(outputs.data, 2) # Shape: (Batch, N)
                
                train_total += labels.numel()
                train_correct += (predicted == labels).sum().item()

            epoch_train_loss = train_loss / len(train_loader.dataset)
            epoch_train_acc = train_correct / train_total

            # --- Validation Phase ---
            model.eval()
            val_loss, val_correct, val_total = 0.0, 0, 0
            
            with torch.no_grad():
                for points, labels in val_loader:
                    points, labels = points.to(device), labels.to(device)
                    
                    outputs = model(points) # Shape: (Batch, N, 2)
                    
                    # Flatten for Loss
                    loss = criterion(outputs.view(-1, 2), labels.view(-1))
                    val_loss += loss.item() * points.size(0)
                    
                    # Predictions (dim=2 because shape is Batch x N x 2)
                    _, predicted = torch.max(outputs.data, 2) # Shape: (Batch, N)
                    
                    val_total += labels.numel()
                    # Now both 'predicted' and 'labels' have shape (Batch, N), so comparison works!
                    val_correct += (predicted == labels).sum().item()
                    
            epoch_val_loss = val_loss / len(val_loader.dataset)
            epoch_val_acc = val_correct / val_total
            
            # MLflow Logging
            mlflow.log_metrics({
                 "train_loss": epoch_train_loss,
                 "train_acc": epoch_train_acc,
                 "val_loss": epoch_val_loss,
                 "val_acc": epoch_val_acc
            }, step=epoch)
            
            print(f"Fold {fold+1} | Epoch {epoch+1}/{epochs} | "
                  f"Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.4f} | "
                  f"Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.4f}")
            
    return model

In [24]:
def visualize_test_predictions(test_files, model, device="cuda"):
    model.eval()
    print(f"Total test files to visualize: {len(test_files)}")

    chunk_size = 30000 
    
    for filepath in test_files:
        print(f"\nProcessing and Visualizing: {filepath}")
        
        ext = filepath.split('.')[-1].lower()
        if ext in ['las', 'laz']:
            las = laspy.read(filepath)
            points = np.vstack((las.x, las.y, las.z)).transpose()
        elif ext == 'pcd':
            pcd = o3d.io.read_point_cloud(filepath)
            points = np.asarray(pcd.points)
        else:
            print(f"Skipping unsupported format: {filepath}")
            continue

        shuffle_idx = np.random.permutation(len(points))
        points = points[shuffle_idx]
        points = points - np.mean(points, axis=0)
        max_dist = np.max(np.sqrt(np.sum(points**2, axis=1)))
        if max_dist > 0:
            points = points / max_dist
            # transpose(0, 1)
        input_tensor = torch.tensor(points, dtype=torch.float32).unsqueeze(0).to(device)
        total_points = input_tensor.size(1)
        
        print(f"Total points in file: {total_points}. Processing in chunks of {chunk_size}...")
        
        all_preds = []
        

        with torch.no_grad():
            for i in range(0, total_points, chunk_size):
              
                # chunk = input_tensor[:, :, i : i + chunk_size]
                chunk = input_tensor[:, i : i + chunk_size, :] 
                outputs = model(chunk)
                preds = torch.argmax(outputs, dim=-1).squeeze(0)
          
                all_preds.append(preds.cpu())
                
       
        final_preds = torch.cat(all_preds).numpy()
        
        # Color Mapping
        colors = np.zeros_like(points)
        colors[final_preds == 0] = [0.7, 0.7, 0.7] # Others (Gray)
        colors[final_preds == 1] = [1.0, 0.0, 0.0] # Wood Powder (Red)
        
        # Open3D Visualization
        pcd_vis = o3d.geometry.PointCloud()
        pcd_vis.points = o3d.utility.Vector3dVector(points-np.mean(points,axis=0))
        pcd_vis.colors = o3d.utility.Vector3dVector(colors)
        
        o3d.visualization.draw_geometries(
            [pcd_vis], 
            window_name=f"PointNet Prediction | {filepath.split('/')[-1]} | Red=Wood, Gray=Others",
            width=1024, 
            height=768,
            point_show_normal=False
        )


# def visualize_test_predictions(test_files, model, device="cuda"):
#     model.eval()
#     print(f"Total test files to visualize: {len(test_files)}")

#     chunk_size = 30000 
    
#     for filepath in test_files:
#         print(f"\nProcessing and Visualizing: {filepath}")
        
#         ext = filepath.split('.')[-1].lower()
#         if ext in ['las', 'laz']:
#             las = laspy.read(filepath)
#             points = np.vstack((las.x, las.y, las.z)).transpose() # Shape: (3, N)
#         elif ext == 'pcd':
#             pcd = o3d.io.read_point_cloud(filepath)
#             points = np.asarray(pcd.points).transpose() # Shape: (3, N)
#         else:
#             print(f"Skipping unsupported format: {filepath}")
#             continue

#         shuffle_idx = np.random.permutation(points.shape[1])
#         points = points[:, shuffle_idx]
#         points = points - np.mean(points, axis=1, keepdims=True)
#         max_dist = np.max(np.sqrt(np.sum(points**2, axis=0)))
#         if max_dist > 0:
#             points = points / max_dist
            
#         # FIX: transpose(0, 1) added to make shape (N, 3), then unsqueeze(0) makes it (1, N, 3)
#         input_tensor = torch.tensor(points, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(device)
#         total_points = input_tensor.size(1) # N
        
#         print(f"Total points in file: {total_points}. Processing in chunks of {chunk_size}...")
        
#         all_preds = []
        
#         with torch.no_grad():
#             for i in range(0, total_points, chunk_size):
#                 # Correctly slicing the N dimension
#                 chunk = input_tensor[:, i : i + chunk_size, :] # Shape: (1, chunk, 3)
                
#                 outputs = model(chunk) # Shape: (1, chunk, 2)
#                 preds = torch.argmax(outputs, dim=-1).squeeze(0) # Shape: (chunk,)
                
#                 all_preds.append(preds.cpu())
                
#         final_preds = torch.cat(all_preds).numpy()
        
#         # Color Mapping
#         colors = np.zeros_like(points.transpose()) # Shape: (N, 3) for colors
#         colors[final_preds == 0] = [0.7, 0.7, 0.7] # Others (Gray)
#         colors[final_preds == 1] = [1.0, 0.0, 0.0] # Wood Powder (Red)
        
#         # Open3D Visualization
#         pcd_vis = o3d.geometry.PointCloud()
#         pcd_vis.points = o3d.utility.Vector3dVector(points.transpose())
#         pcd_vis.colors = o3d.utility.Vector3dVector(colors)
        
#         o3d.visualization.draw_geometries(
#             [pcd_vis], 
#             window_name=f"Point Transformer Prediction | {filepath.split('/')[-1]} | Red=Wood, Gray=Others",
#             width=1024, 
#             height=768,
#             point_show_normal=False
#         )

In [26]:
import glob
if __name__=="__main__":
    data_dir="data"

    all_files = []
    for ext in ['*.las', '*.laz', '*.pcd']:
        all_files.extend(glob.glob(os.path.join(data_dir, ext)))

    train_val_data,test_data=train_test_split(all_files,test_size=.2,random_state=42)
    test_dataset=WoodPowderDataset(test_data,num_points=16384)
    test_loader=DataLoader(test_dataset,batch_size=2,shuffle=False)
    k_folds=5
    kf=KFold(n_splits=k_folds,shuffle=True,random_state=42)
    train_val_dataset=WoodPowderDataset(train_val_data,num_points=16384)

    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("Point Transformer V3")
    with mlflow.start_run(run_name="K-fold-base-run"):
        mlflow.log_params({"k_folds":10,"test_size":0.2})


        # for fold,(train_idx,val_idx) in enumerate(kf.split(train_val_dataset)):
        # for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(train_val_dataset)))):
        for fold, (train_idx, val_idx) in enumerate(kf.split(train_val_data)):
            train_subset=Subset(train_val_dataset,train_idx)
            val_subset=Subset(train_val_dataset,val_idx)
            train_loader=DataLoader(train_subset,batch_size=2,shuffle=True)
            val_loader=DataLoader(val_subset,shuffle=False,batch_size=2)
            # model=PointNetClassifier(num_classes=2)
            # model = PointNetSegmentation(num_classes=2)
            model=PointTransformerSeg(3,2)
            # if torch.cuda.device_count()>1:
                # model=nn.DataParallel(model)
                # n_gpus = min(torch.cuda.device_count(), train_loader.batch_size)
                # model = nn.DataParallel(model, device_ids=list(range(n_gpus)))
            model=model.to(device)
            trained_model = train_evaluate_fold(fold, model, train_loader, val_loader, epochs=100, lr=0.0001, device=device)
            visualize_test_predictions(test_data, trained_model, device=device)
    

🏃 View run Fold-1 at: http://127.0.0.1:5000/#/experiments/2/runs/4bfafb317a7e48ddbb302e8f852f18a8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run K-fold-base-run at: http://127.0.0.1:5000/#/experiments/2/runs/26c92e2ad8bc48e09eb66d21773a7fed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


TypeError: PointTransformerSeg.forward() missing 1 required positional argument: 'pos'